# 17: Split the Kaggle test set, then run inference (label-free)

The 26,788 new patients are being used to make **error dependence estimable**, which is now the
binding constraint: the old validation split gave 1.84 shared missed PDR eyes per pair and
reliability 0.41, so partner selection was impossible. The v3 primary is already confirmed on the
fresh set, and the fresh set is spent as the *outcome* for the selection question (notebook 14
used it that way), so this set supplies both the estimation and the confirmation sample.

**Cell 2 fixes the split before any analysis**, patient-disjoint, label-free, seeded:
- `estimation` (2/3 of patients): where dependence and every candidate selection rule is measured
- `confirmation` (1/3): touched once, later, to test whether selection actually works

That gives roughly 800 PDR eyes for estimation (a 14-fold increase over 57) and 400 for
confirmation, more than the fresh set had when it detected the primary.

**Cells 3-5 run all 33 checkpoints over all 53,576 images and write predictions only.** No
metrics, no AUC, nothing evaluated, so this can safely precede the lock for the selection test.

A100. About 2 to 2.5 hours. Resumable. Needs the `HF_TOKEN` secret. Run cells 1-5.

In [1]:
# Cell 1: auth, deps, Drive helpers
!pip -q install timm transformers
import io, os, json, time, zipfile, glob, datetime
import numpy as np, pandas as pd, torch
import torch.nn as nn
from google.colab import auth, userdata
auth.authenticate_user()
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
svc=gbuild('drive','v3')

F_CHECKPOINTS='1DAAXJ2fjk2zAzx7PpGaWsk4BXudeA5Le'
F_PREDICTIONS='1nCloJ5o9veZb2xJL_2EjdVHQXN31jy4w'
F_SPLITS='13x_CS7N_UasGRSuNYMyD5To2RTYjL-EP'
F_AUDIT='1zuvfcxC3umHAq_uf7Vx9omMEgUEuQhTt'
RETFOUND_REPO='iszt/RETFound_mae_meh'
NUM_WORKERS=min(8, os.cpu_count())
ARCHS=['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
       'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
       'retfound_binary','retfound_5class','retfound_adversarial']
SEEDS=[0,1,2]
def tag(k): return '' if k==0 else f'_seed{k}'

def dl(fid,p):
    with io.FileIO(p,'wb') as fh:
        d=MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid), chunksize=64*1024*1024); done=False
        while not done: _,done=d.next_chunk()
def find_id(n,f):
    r=svc.files().list(q=f"'{f}' in parents and name='{n}' and trashed=false", fields='files(id)').execute().get('files',[])
    return r[0]['id'] if r else None
def list_folder(f):
    out,tok=[],None
    while True:
        r=svc.files().list(q=f"'{f}' in parents and trashed=false", fields='files(id,name),nextPageToken',
                           pageToken=tok, pageSize=1000).execute()
        out+=r.get('files',[]); tok=r.get('nextPageToken')
        if not tok: break
    return out
def up(p,f,m='text/csv'):
    svc.files().create(body={'name':os.path.basename(p),'parents':[f]},
                       media_body=MediaFileUpload(p,mimetype=m,resumable=True), fields='id').execute()
    print('  uploaded', os.path.basename(p))

v2=svc.files().list(q="name='dr_ensemble_v2' and mimeType='application/vnd.google-apps.folder' and trashed=false",
                    fields='files(id)').execute()['files'][0]['id']
F_KT_IMAGES=[f['id'] for f in list_folder(v2) if f['name']=='07_kaggle_test_images'][0]
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

GPU: NVIDIA A100-SXM4-40GB


In [2]:
# Cell 2: fix the estimation / confirmation split (patient-disjoint, label-free, seeded)
SPLIT_NAME='eyepacs_split_kaggletest_estconf_v2.csv'
existing=find_id(SPLIT_NAME, F_SPLITS)
if existing:
    dl(existing, SPLIT_NAME); kt=pd.read_csv(SPLIT_NAME)
    print('split already fixed; reusing it')
else:
    dl(find_id('eyepacs_split_kaggletest_v2.csv', F_SPLITS), 'kt.csv')
    kt=pd.read_csv('kt.csv')
    SEED=20260923; rng=np.random.default_rng(SEED)
    kt['patient_id']=kt['patient_id'].astype(str)
    pats=np.array(sorted(kt.patient_id.unique())); rng.shuffle(pats)
    n_est=int(round(len(pats)*2/3))
    assign={p:('estimation' if i<n_est else 'confirmation') for i,p in enumerate(pats)}
    kt['pool']=kt['patient_id'].map(assign)
    a=set(kt.loc[kt.pool=='estimation','patient_id']); b=set(kt.loc[kt.pool=='confirmation','patient_id'])
    assert not (a&b), 'patient overlap between pools'
    kt.to_csv(SPLIT_NAME, index=False); up(SPLIT_NAME, F_SPLITS)
    print(f'split fixed with seed {SEED}')
kt['image_id']=kt['image_id'].astype(str); kt['patient_id']=kt['patient_id'].astype(str)
print('kaggle test:', len(kt), 'images |', kt.patient_id.nunique(), 'patients')
for pool,g in kt.groupby('pool'):
    print(f"  {pool}: {len(g)} images, {g.patient_id.nunique()} patients, PDR {int((g.level==4).sum())}, "
          f"grade3 {int((g.level==3).sum())}, referable {g.binary_label.mean():.4f}")
print('\nfor comparison: old validation split had 57 PDR eyes; the fresh set has 328')

  uploaded eyepacs_split_kaggletest_estconf_v2.csv
split fixed with seed 20260923
kaggle test: 53576 images | 26788 patients
  confirmation: 17858 images, 8929 patients, PDR 382, grade3 390, referable 0.1941
  estimation: 35718 images, 17859 patients, PDR 824, grade3 824, referable 0.1908

for comparison: old validation split had 57 PDR eyes; the fresh set has 328


In [3]:
# Cell 3: fetch images
IMG='/content/kt'; os.makedirs(IMG, exist_ok=True)
if len(glob.glob(f'{IMG}/*.png'))<len(kt):
    for f in sorted(list_folder(F_KT_IMAGES), key=lambda x:x['name']):
        if not f['name'].endswith('.zip'): continue
        z=f"/content/{f['name']}"
        if not os.path.exists(z):
            print('downloading', f['name'], flush=True); dl(f['id'], z)
        with zipfile.ZipFile(z) as zf: zf.extractall(IMG)
        os.remove(z); print('  extracted', f['name'], flush=True)
INDEX={os.path.splitext(n)[0]: f'{IMG}/{n}' for n in os.listdir(IMG) if n.endswith('.png')}
missing=sorted(set(kt['image_id'])-set(INDEX))
print(f'images on disk: {len(INDEX)} | required: {len(kt)} | missing: {len(missing)}')
if missing: raise SystemExit(f'missing images, e.g. {missing[:5]}')

downloading eyepacs_kaggletest_batch1_v2.zip
  extracted eyepacs_kaggletest_batch1_v2.zip
downloading eyepacs_kaggletest_batch2_v2.zip
  extracted eyepacs_kaggletest_batch2_v2.zip
downloading eyepacs_kaggletest_batch3_v2.zip
  extracted eyepacs_kaggletest_batch3_v2.zip
downloading eyepacs_kaggletest_batch4_v2.zip
  extracted eyepacs_kaggletest_batch4_v2.zip
images on disk: 53576 | required: 53576 | missing: 0


In [4]:
# Cell 4: skeletons, strict loading, predict
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoConfig, AutoModel
from PIL import Image
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
TF=transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
                       transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
class DS(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,i): return TF(Image.open(INDEX[self.df.at[i,'image_id']]).convert('RGB')), i
class RETFound(nn.Module):
    def __init__(self, nc):
        super().__init__()
        cfg=AutoConfig.from_pretrained(RETFOUND_REPO)
        self.base=AutoModel.from_config(cfg, add_pooling_layer=False)
        self.fc_norm=nn.LayerNorm(cfg.hidden_size, eps=1e-6); self.classifier=nn.Linear(cfg.hidden_size, nc)
    def forward(self,x):
        h=self.base(pixel_values=x).last_hidden_state
        return self.classifier(self.fc_norm(h[:,1:].mean(dim=1)))
def load_model(a,k,dev):
    ck=f'eyepacs_{a}{tag(k)}_v2.pt'; cid=find_id(ck, F_CHECKPOINTS)
    if not cid: raise FileNotFoundError(ck)
    dl(cid, ck); blob=torch.load(ck, map_location='cpu', weights_only=False); cfg=blob['config']
    m=RETFound(cfg['nc']) if cfg['arch']=='retfound' else timm.create_model(cfg['arch'], pretrained=False, num_classes=cfg['nc'])
    m.load_state_dict(blob['state_dict'], strict=True); os.remove(ck)
    return m.to(dev).eval(), cfg
@torch.no_grad()
def predict(model, dev, bs):
    dl_=DataLoader(DS(kt), batch_size=bs, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    probs=np.zeros(len(kt), dtype=np.float64)
    for x,idx in dl_:
        out=model(x.to(dev, non_blocking=True)).float()
        p=torch.sigmoid(out[:,0]) if out.shape[1]==1 else torch.softmax(out,1)[:,2:].sum(1)
        probs[idx.numpy()]=p.double().cpu().numpy()
    return probs
print('ready')

ready


In [5]:
# Cell 5: run all 33 models; predictions only, no metrics
dev='cuda' if torch.cuda.is_available() else 'cpu'
have={f['name'] for f in list_folder(F_PREDICTIONS)}
todo=[(a,k) for a in ARCHS for k in SEEDS if f'eyepacs_predictions_kaggletest_{a}{tag(k)}_v2.csv' not in have]
print(f'to run: {len(todo)} of 33')
log=[]
for i,(a,k) in enumerate(todo,1):
    label=a+(tag(k) or '')
    print(f'\n=== [{i}/{len(todo)}] {label} ===', flush=True)
    t0=time.time()
    try:
        model,cfg=load_model(a,k,dev)
    except Exception as e:
        print('  FAILED to load:', e); log.append({'model':label,'status':f'load failed: {e}'}); continue
    pr=predict(model, dev, 32 if cfg['arch']=='retfound' else 128)
    out=f'eyepacs_predictions_kaggletest_{a}{tag(k)}_v2.csv'
    pd.DataFrame({'image_id':kt['image_id'],'patient_id':kt['patient_id'],'eye':kt['eye'],
                  'dataset':'eyepacs','split':'kaggletest','pool':kt['pool'],
                  'true_grade':kt['level'].astype(int),'true_label':kt['binary_label'].astype(int),
                  'predicted_probability':pr,'predicted_class':(pr>=0.5).astype(int),
                  'model_name':a,'replicate':k}).to_csv(out, index=False, float_format='%.17g')
    up(out, F_PREDICTIONS)
    log.append({'model':label,'status':'ok','minutes':round((time.time()-t0)/60,1),
                'mean_prob':round(float(pr.mean()),4)})
    print(f"  done in {log[-1]['minutes']} min (mean predicted probability {log[-1]['mean_prob']}; "
          f"no metrics computed)", flush=True)
    del model; torch.cuda.empty_cache()
done={f['name'] for f in list_folder(F_PREDICTIONS)}
got=sum(f'eyepacs_predictions_kaggletest_{a}{tag(k)}_v2.csv' in done for a in ARCHS for k in SEEDS)
rec={'generated_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'files_present':got,
     'split_file':SPLIT_NAME,'log':log,
     'note':'Predictions only. The confirmation pool must not be analysed until the selection test is locked.'}
with open('kaggletest_inference_log_v2.json','w') as f: json.dump(rec,f,indent=2)
up('kaggletest_inference_log_v2.json', F_AUDIT, 'application/json')
print(f'\nkaggle-test prediction files: {got}/33')

to run: 33 of 33

=== [1/33] densenet121_binary ===
  uploaded eyepacs_predictions_kaggletest_densenet121_binary_v2.csv
  done in 1.9 min (mean predicted probability 0.1725; no metrics computed)

=== [2/33] densenet121_binary_seed1 ===
  uploaded eyepacs_predictions_kaggletest_densenet121_binary_seed1_v2.csv
  done in 1.8 min (mean predicted probability 0.1907; no metrics computed)

=== [3/33] densenet121_binary_seed2 ===
  uploaded eyepacs_predictions_kaggletest_densenet121_binary_seed2_v2.csv
  done in 1.9 min (mean predicted probability 0.1689; no metrics computed)

=== [4/33] densenet121_5class ===
  uploaded eyepacs_predictions_kaggletest_densenet121_5class_v2.csv
  done in 1.9 min (mean predicted probability 0.1818; no metrics computed)

=== [5/33] densenet121_5class_seed1 ===
  uploaded eyepacs_predictions_kaggletest_densenet121_5class_seed1_v2.csv
  done in 1.9 min (mean predicted probability 0.1834; no metrics computed)

=== [6/33] densenet121_5class_seed2 ===
  uploaded eyepa

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

  uploaded eyepacs_predictions_kaggletest_retfound_binary_v2.csv
  done in 2.6 min (mean predicted probability 0.142; no metrics computed)

=== [26/33] retfound_binary_seed1 ===
  uploaded eyepacs_predictions_kaggletest_retfound_binary_seed1_v2.csv
  done in 2.9 min (mean predicted probability 0.1568; no metrics computed)

=== [27/33] retfound_binary_seed2 ===
  uploaded eyepacs_predictions_kaggletest_retfound_binary_seed2_v2.csv
  done in 2.6 min (mean predicted probability 0.1807; no metrics computed)

=== [28/33] retfound_5class ===
  uploaded eyepacs_predictions_kaggletest_retfound_5class_v2.csv
  done in 2.6 min (mean predicted probability 0.1804; no metrics computed)

=== [29/33] retfound_5class_seed1 ===
  uploaded eyepacs_predictions_kaggletest_retfound_5class_seed1_v2.csv
  done in 2.6 min (mean predicted probability 0.1705; no metrics computed)

=== [30/33] retfound_5class_seed2 ===
  uploaded eyepacs_predictions_kaggletest_retfound_5class_seed2_v2.csv
  done in 2.6 min (mean